# V-JEPA 2 ViT-L × HATREC — Deep Analysis

Phân tích benchmark frozen V-JEPA 2 embedding + Logistic Regression trên split giữ riêng toàn bộ cycle. Notebook phân biệt rõ **classification performance** với **video reasoning/motion understanding**.

In [ ]:
from pathlib import Path
import json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm, binomtest
from sklearn.metrics import classification_report, confusion_matrix
from IPython.display import display, Markdown
sns.set_theme(style='whitegrid', context='notebook'); plt.rcParams['figure.dpi']=120

cwd=Path.cwd(); direct=cwd/'hatrec_vjepa2_metrics.json'
if direct.exists(): DATA_DIR=cwd
else:
    found=list(cwd.rglob('vjepa2_hatrec_results/hatrec_vjepa2_metrics.json'))
    assert found,'Không tìm thấy vjepa2_hatrec_results'; DATA_DIR=found[0].parent
OUT=DATA_DIR/'deep_analysis'; OUT.mkdir(exist_ok=True)
metrics=json.loads((DATA_DIR/'hatrec_vjepa2_metrics.json').read_text(encoding='utf-8'))
pred=pd.read_csv(DATA_DIR/'hatrec_vjepa2_predictions.csv')
TASKS={0:'Assembling spring',1:'White plastic',2:'Screwing-1',3:'Inflating valve',4:'Black plastic',5:'Screwing-2',6:'Fixing cable'}
print('DATA_DIR:',DATA_DIR); display(pred.head())

## 1. Integrity, split và leakage safeguards

In [ ]:
split=metrics['split']
audit=pd.Series({
 'dataset_videos':split['videos'], 'train_videos':split['rows']['train'], 'val_videos':split['rows']['val'], 'test_videos':split['rows']['test'],
 'train_cycles':split['train_cycles'], 'val_cycles':split['val_cycles'], 'test_cycles':split['test_cycles'],
 'test_rows_loaded':len(pred), 'test_cycles_loaded':pred.cycle.nunique(), 'duplicate_test_ids':pred.neutral_id.duplicated().sum(),
 'physical_filename_neutralization':split['physical_filename_neutralization'], 'exact_video_duplicates':split['exact_duplicates']
},name='value')
display(audit.to_frame())
assert len(pred)==84 and pred.cycle.nunique()==12 and pred.neutral_id.nunique()==84
assert pred.groupby('label').size().eq(12).all() and split['exact_duplicates']==0 and split['physical_filename_neutralization']
print('Procedural leakage checks: PASS')

## 2. Main results, ablations và negative controls

In [ ]:
rows=[]
for key,label,kind in [
 ('cycle_holdout_full64','Cycle holdout — 64 frames','primary'),('cycle_holdout_visible14','Cycle holdout — 14 visible frames','ablation'),
 ('random_clip_full64','Random clip — 64 frames','leakage-risk comparator'),('blind_majority','Majority blind','negative control'),
 ('shuffled_label_control','Shuffled labels','negative control')]:
    v=metrics[key]; rows.append({'setting':label,'kind':kind,'accuracy':v['accuracy'],'macro_f1':v['macro_f1']})
result_table=pd.DataFrame(rows)
display(result_table.style.format({'accuracy':'{:.2%}','macro_f1':'{:.2%}'}))
fig,ax=plt.subplots(1,2,figsize=(15,5))
sns.barplot(data=result_table,x='accuracy',y='setting',hue='kind',dodge=False,ax=ax[0]);ax[0].axvline(1/7,ls='--',c='black',label='chance');ax[0].set_xlim(0,1.02);ax[0].set_title('Accuracy by experimental setting')
sns.barplot(data=result_table,x='macro_f1',y='setting',hue='kind',dodge=False,ax=ax[1]);ax[1].axvline(1/7,ls='--',c='black');ax[1].set_xlim(0,1.02);ax[1].set_title('Macro-F1 by experimental setting')
for a in ax:
    if a.legend_: a.legend_.remove()
plt.tight_layout();plt.savefig(OUT/'01_results_and_controls.png',bbox_inches='tight');plt.show()
result_table.to_csv(OUT/'experimental_settings.csv',index=False)

## 3. Uncertainty: perfect score vẫn không có nghĩa true accuracy = 100%

In [ ]:
def wilson(k,n,alpha=.05):
    z=norm.ppf(1-alpha/2);p=k/n;den=1+z*z/n;center=(p+z*z/(2*n))/den;half=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/den
    return center-half,center+half
k=int(pred.correct.sum());n=len(pred);lo,hi=wilson(k,n)
p_chance=binomtest(k,n,p=1/7,alternative='greater').pvalue
uncertainty=pd.DataFrame([
 {'metric':'Test accuracy','estimate':k/n,'low_95':lo,'high_95':hi},
 {'metric':'Cluster bootstrap Macro-F1','estimate':metrics['cycle_holdout_full64']['macro_f1'],'low_95':metrics['cycle_bootstrap_macro_f1_95ci'][0],'high_95':metrics['cycle_bootstrap_macro_f1_95ci'][1]}])
display(uncertainty.style.format({'estimate':'{:.2%}','low_95':'{:.2%}','high_95':'{:.2%}'}))
print(f'Exact binomial p-value versus 1/7 chance: {p_chance:.3e}')
print(f'Interpretation: 84/84 implies Wilson 95% CI {lo:.2%}–{hi:.2%}, not guaranteed 100% population accuracy.')
uncertainty.to_csv(OUT/'uncertainty_intervals.csv',index=False)

## 4. Per-class và per-cycle stability

In [ ]:
labels=list(range(7));names=[TASKS[i] for i in labels]
report=pd.DataFrame(classification_report(pred.label,pred.prediction,labels=labels,target_names=names,output_dict=True,zero_division=0)).T
cm=confusion_matrix(pred.label,pred.prediction,labels=labels)
cycle_class=pd.crosstab(pred.cycle,pred.label,values=pred.correct,aggfunc='mean').reindex(columns=labels)
fig,ax=plt.subplots(1,2,figsize=(18,6))
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=names,yticklabels=names,ax=ax[0]);ax[0].set_title('Cycle-held-out confusion matrix');ax[0].tick_params(axis='x',rotation=60)
sns.heatmap(cycle_class,annot=True,fmt='.0%',cmap='RdYlGn',vmin=0,vmax=1,xticklabels=names,ax=ax[1]);ax[1].set_title('Correctness by held-out cycle × task');ax[1].tick_params(axis='x',rotation=60)
plt.tight_layout();plt.savefig(OUT/'02_class_cycle_stability.png',bbox_inches='tight');plt.show()
display(report);report.to_csv(OUT/'per_class_report.csv');cycle_class.to_csv(OUT/'cycle_task_correctness.csv')

## 5. Similarity audit và shortcut risk

In [ ]:
cos=metrics['test_to_train_cosine']
diagnostics=pd.DataFrame([
 ['Exact byte duplicate across all videos',split['exact_duplicates'],'PASS','No exact duplicates'],
 ['Label-bearing filenames visible to encoder',not split['physical_filename_neutralization'],'PASS','Files were physically neutralized'],
 ['Max cosine > 0.999',cos['above_0_999'],'PASS' if cos['above_0_999']==0 else 'REVIEW','Embedding near-duplicate threshold'],
 ['Mean max test→train cosine',cos['mean_max'],'REVIEW','High semantic/visual similarity'],
 ['64-frame versus 14-frame performance gap',metrics['cycle_holdout_full64']['macro_f1']-metrics['cycle_holdout_visible14']['macro_f1'],'INCONCLUSIVE','No degradation with 14 visible frames'],
 ['Single-frame repeated control',np.nan,'MISSING','Required to separate static cues from motion'],
 ['Temporal-shuffle control',np.nan,'MISSING','Required to test order sensitivity'],
],columns=['check','value','status','interpretation'])
display(diagnostics)
fig,ax=plt.subplots(figsize=(9,4));ax.bar(['mean max','p95 max'],[cos['mean_max'],cos['p95_max']],color=['#F28E2B','#E15759']);ax.axhline(.999,ls='--',c='black',label='near-duplicate threshold');ax.set_ylim(.95,1.001);ax.set_ylabel('Cosine similarity');ax.set_title('Nearest train embedding for held-out test clips');ax.legend();plt.tight_layout();plt.savefig(OUT/'03_similarity_audit.png',bbox_inches='tight');plt.show()
diagnostics.to_csv(OUT/'leakage_shortcut_diagnostics.csv',index=False)

## 6. Calibration và capability boundary

In [ ]:
cal=pd.DataFrame([
 ['ECE',metrics['cycle_holdout_full64']['ece'],'Lower is better'],
 ['Multiclass Brier',metrics['cycle_holdout_full64']['multiclass_brier'],'Lower is better']
],columns=['metric','value','direction'])
display(cal.style.format({'value':'{:.6f}'}))
display(pd.DataFrame([{'capability':k,'claim':v} for k,v in metrics['capability_mapping'].items()]))
print('Reliability diagram cannot be reconstructed: per-sample probabilities were not saved in predictions.csv.')
print('This benchmark measures B1 task/state classification only. It does not test language reasoning, temporal segmentation, causal explanation or OOD abstention.')

## 7. Research verdict và next experiments

In [ ]:
summary=f'''# V-JEPA 2 × HATREC: research verdict

- Dataset: 546 videos, 78 cycles, 7 balanced tasks. Split: 54 train / 12 validation / 12 test cycles.
- Primary result: 84/84 correct on held-out cycles; Macro-F1 = 100%. Wilson 95% accuracy interval = {lo:.2%}–{hi:.2%}.
- 14-visible-frame ablation also reaches 100%; the additional frames did not improve this benchmark.
- Negative controls behave near chance: majority accuracy {metrics['blind_majority']['accuracy']:.2%}; shuffled-label accuracy {metrics['shuffled_label_control']['accuracy']:.2%}.
- Procedural leakage checks pass: filenames neutralized and exact duplicates = 0.
- However, mean nearest train cosine is {cos['mean_max']:.4f} and p95 is {cos['p95_max']:.4f}; the environment/tasks remain visually similar.
- Therefore this result validates linear separability of HATREC task representations, not language reasoning and not yet temporal motion understanding.

## Required next tests
1. Single middle frame repeated to 64 frames.
2. Temporal frame-order shuffle/reversal.
3. Object/tool masking or background intervention.
4. Hold out operator/workstation/product if metadata becomes available.
5. Save per-sample probabilities for reliability diagrams and abstention analysis.
'''
(OUT/'VJEPA2_HATREC_RESEARCH_VERDICT.md').write_text(summary,encoding='utf-8')
display(Markdown(summary));print('Saved:',OUT)